# Lesson 1: Psychophysical Modelling

## The Noisy Logarithmic Coding (NLC) model

When we judge quantities — the number of coins in a pile, the size of a reward — our
internal representations are noisy.  The **NLC model** posits that the brain encodes
numerical magnitude $n$ on a logarithmic scale, and that this log-representation is
corrupted by Gaussian noise:

$$r \sim \mathcal{N}(\log n, \; \nu^2)$$

Given two stimuli $n_1$ and $n_2$ with independent noise, the probability of choosing
$n_2$ as the larger is

$$P(\text{chose}\; n_2) = \Phi\!\left(\frac{\log(n_2/n_1)}{\sqrt{\nu_1^2 + \nu_2^2}}\right)$$

where $\Phi$ is the standard normal CDF, $\nu_1$ is the noise on $n_1$ (the first-presented
option) and $\nu_2$ is the noise on $n_2$ (the second-presented option).

In many experimental paradigms stimuli are shown **sequentially**: the observer perceives
$n_1$, holds it in working memory, then perceives $n_2$.  Memory retention may add extra
noise on top of perception, so the model allows $\nu_1 \neq \nu_2$.  In tasks where both
options are visible simultaneously there is no reason to separate the two, and a single
shared $\nu$ is sufficient.

**Scale invariance**: when $x = \log(n_2/n_1)$ is used as the horizontal axis, the
psychometric function collapses to a single sigmoid regardless of the absolute magnitude
of $n_1$ — a direct prediction of the logarithmic encoding.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm as scipy_norm

# ── Scale invariance demo ─────────────────────────────────────────────────────
nu = 0.45                              # equal noise for both options
n1_vals = [5, 10, 20, 28]
n2_linear = np.linspace(1, 60, 300)
log_ratios = np.linspace(-1.8, 1.8, 300)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pal = sns.color_palette('Blues_d', len(n1_vals))

for (n1, c) in zip(n1_vals, pal):
    p_lin = scipy_norm.cdf(np.log(n2_linear / n1) / (np.sqrt(2) * nu))
    p_log = scipy_norm.cdf(log_ratios            / (np.sqrt(2) * nu))
    axes[0].plot(n2_linear, p_lin, color=c, lw=2, label=f'n1={n1}')
    axes[1].plot(log_ratios, p_log, color=c, lw=2, label=f'n1={n1}')

for ax in axes:
    ax.axhline(0.5, ls='--', c='gray', lw=1)
    ax.set_ylim(-0.03, 1.03)
    ax.legend(title='n1', fontsize=8)
    sns.despine(ax=ax)

axes[0].axvline(0, ls='--', c='gray', lw=1)
axes[1].axvline(0, ls='--', c='gray', lw=1)
axes[0].set_xlabel('n2 (linear)')
axes[1].set_xlabel('log(n2 / n1)')
axes[0].set_ylabel('P(chose n2)')
axes[0].set_title('Linear scale — curves spread out')
axes[1].set_title('Log-ratio scale — curves collapse')
plt.tight_layout()

### Effect of noise level and asymmetric noise

**Precision** $\gamma = 1/(\sqrt{2}\,\nu)$ is the slope of the psychometric curve on a
log-ratio axis.  Higher noise → shallower curve → less precise observer.

When stimuli are presented sequentially, $n_1$ must be retained in memory while $n_2$ is
being perceived, which often leads to $\nu_1 > \nu_2$.  Unequal noise alone (without a
prior) does **not** shift the crossing point — it only changes the effective slope.  The
shift arises from the Bayesian prior, shown next.

In [ ]:
log_r = np.linspace(-2, 2, 300)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: noise level → precision
ax = axes[0]
for nu_val, label, c in [(.2, 'ν = 0.20 (sharp)', '#08519c'),
                           (.5, 'ν = 0.50', '#3182bd'),
                           (1., 'ν = 1.00 (noisy)', '#bdd7e7')]:
    ax.plot(log_r, scipy_norm.cdf(log_r / (np.sqrt(2)*nu_val)), lw=2, color=c, label=label)
ax.axhline(.5, ls='--', c='gray', lw=1); ax.axvline(0, ls='--', c='gray', lw=1)
ax.set_xlabel('log(n2/n1)'); ax.set_ylabel('P(chose n2)')
ax.set_title('Effect of noise level on precision')
ax.legend(); sns.despine(ax=ax)

# Right: asymmetric noise  ν1 vs ν2
ax = axes[1]
for (nu1, nu2), label, c in [((0.4, 0.4), 'Equal noise  ν₁=ν₂=0.4', '#1a9850'),
                               ((0.7, 0.3), 'Asymm. noise  ν₁=0.7, ν₂=0.3', '#d73027')]:
    ax.plot(log_r, scipy_norm.cdf(log_r / np.sqrt(nu1**2+nu2**2)), lw=2, color=c, label=label)
ax.axhline(.5, ls='--', c='gray', lw=1); ax.axvline(0, ls='--', c='gray', lw=1)
ax.set_xlabel('log(n2/n1)')
ax.set_title('Asymmetric noise (same total) → different slope')
ax.legend(fontsize=8); sns.despine(ax=ax)

plt.tight_layout()

### The Bayesian prior and central tendency bias

Pure measurement noise is symmetric.  The *shift* of the psychometric curve arises
from a **Bayesian prior** over log-magnitudes.  The observer combines noisy evidence
$r \sim \mathcal{N}(\log n, \nu^2)$ with a prior $\mathcal{N}(\mu_0, \sigma_0^2)$ to
form a posterior:

$$\hat\mu = \underbrace{\frac{\sigma_0^2}{\sigma_0^2 + \nu^2}}_{\gamma}\, r
           + (1-\gamma)\,\mu_0, \qquad
\hat\sigma^2 = \frac{\sigma_0^2\,\nu^2}{\sigma_0^2 + \nu^2}$$

The posterior mean is pulled toward $\mu_0$ — the **central tendency effect**.  A
stronger prior (smaller $\sigma_0$) compresses representations more, shifting choices
systematically toward the prior mean.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

x = np.linspace(0.5, 5.5, 400)       # log-magnitude axis
log_n_true  = np.log(20)              # true stimulus: n=20
prior_mu    = np.log(10)              # prior centred at n=10
nu          = 0.4                     # measurement noise

for ax, (prior_sd, title) in zip(axes,
        [(1.0, 'Weak prior  (σ₀ = 1.0)'),
         (0.25, 'Strong prior  (σ₀ = 0.25)')]):

    gamma   = prior_sd**2 / (prior_sd**2 + nu**2)
    post_mu = prior_mu + gamma * (log_n_true - prior_mu)
    post_sd = np.sqrt(prior_sd**2 * nu**2 / (prior_sd**2 + nu**2))

    like  = scipy_norm.pdf(x, log_n_true, nu)
    prior = scipy_norm.pdf(x, prior_mu,  prior_sd)
    post  = scipy_norm.pdf(x, post_mu,   post_sd)
    mx    = max(like.max(), prior.max(), post.max())

    ax.fill_between(x, prior/mx, alpha=.3, color='#4393c3', label=f'Prior  μ₀={prior_mu:.2f}')
    ax.fill_between(x, like/mx,  alpha=.3, color='#d6604d', label=f'Evidence  log(n)={log_n_true:.2f}')
    ax.fill_between(x, post/mx,  alpha=.5, color='#4dac26', label=f'Posterior  μ̂={post_mu:.2f}')
    ax.axvline(log_n_true, ls='--', c='#d6604d', lw=1.5)
    ax.axvline(prior_mu,   ls='--', c='#4393c3', lw=1.5)
    ax.axvline(post_mu,    ls='-',  c='#4dac26', lw=2.5)
    ax.set_xlabel('Internal log-magnitude representation')
    ax.set_ylabel('Probability density (normalised)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

plt.suptitle('Central tendency: posterior mean is pulled toward the prior', y=1.02)
plt.tight_layout()

## Barreto-García et al. (2023): Magnitude comparison task

64 participants viewed two sequentially presented coin clouds and judged which contained
more 1-CHF coins.  Reference magnitudes $n_1 \in \{5, 7, 10, 14, 20, 28\}$; comparison
magnitudes $n_2$ varied widely.  We load the bundled dataset and inspect it.

In [ ]:
import pandas as pd
import arviz as az
from bauer.utils.data import load_garcia2022
from bauer.models import MagnitudeComparisonModel

data = load_garcia2022(task='magnitude')
print(f"Subjects: {data.index.get_level_values('subject').nunique()},  "
      f"Trials: {len(data)}")
data.head()

In [ ]:
# Compute log-ratio for each trial
data['log(n2/n1)'] = np.log(data['n2'] / data['n1'])
data['bin'] = (pd.cut(data['log(n2/n1)'], bins=12)
                 .apply(lambda x: x.mid).astype(float))

grouped = (data.groupby(['n1', 'bin'])['choice']
               .agg(['mean', 'count']).reset_index()
               .query('count >= 5'))

g = sns.FacetGrid(grouped, col='n1', col_wrap=3, height=3.2, aspect=1.1)
g.map(plt.scatter, 'bin', 'mean', s=25, alpha=.8, color='#2166ac')
g.map(sns.lineplot,  'bin', 'mean', color='#2166ac', lw=1.5)
for ax in g.axes.flat:
    ax.axhline(.5, ls='--', c='gray', lw=1)
    ax.axvline(.0, ls='--', c='gray', lw=1)
    ax.set_ylim(-.05, 1.05)
g.set_axis_labels('log(n2 / n1)', 'P(chose n2)')
g.set_titles('n1 = {col_name}')
plt.suptitle('Group-averaged psychometric curves — scale invariance confirmed', y=1.02)
plt.tight_layout()

## Fitting `MagnitudeComparisonModel`

We fit the full hierarchical NLC model.  Free parameters per subject:
- **`n1_evidence_sd`** ($\nu_1$): noise on the first-presented cloud (perception + memory retention)
- **`n2_evidence_sd`** ($\nu_2$): noise on the second-presented cloud (perception only)

The group-level means (`_mu`) and standard deviations (`_sd`) are estimated jointly.

In [ ]:
model_mag = MagnitudeComparisonModel(paradigm=data)
model_mag.build_estimation_model(data=data, hierarchical=True, save_p_choice=True)
idata_mag = model_mag.sample(draws=200, tune=200, chains=4, progressbar=True)

In [ ]:
az.plot_posterior(
    idata_mag,
    var_names=['n1_evidence_sd_mu', 'n2_evidence_sd_mu'],
    figsize=(9, 3),
)
plt.suptitle('Group-level noise posteriors  (ν₁ = first option,  ν₂ = second option)', y=1.04)
plt.tight_layout()

In [ ]:
# Subject-level posterior means: ν₁ vs ν₂
params = ['n1_evidence_sd', 'n2_evidence_sd']
means  = (idata_mag.posterior[params]
                   .mean(dim=['chain', 'draw'])
                   .to_dataframe()
                   .reset_index())

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(means['n2_evidence_sd'], means['n1_evidence_sd'],
           alpha=.7, color='steelblue', s=40, zorder=3)
lim = max(means[params].max()) * 1.15
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='ν₁ = ν₂')
ax.set_xlabel('Second-option noise  ν₂  (n2_evidence_sd)')
ax.set_ylabel('First-option noise  ν₁  (n1_evidence_sd)')
ax.set_title('Subject-level noise estimates  (ν₁ > ν₂ for most subjects, consistent with memory retention)')
ax.legend(); sns.despine(); plt.tight_layout()

## Posterior predictive check

We draw predicted choice probabilities from the full posterior and overlay the 95 %
credible interval on the observed group-average data (one panel per $n_1$ value).
Good model fit means the shaded region covers the observed dots.

In [ ]:
ppc_df = model_mag.ppc(data, idata_mag, var_names=['p'])
ppc_p  = ppc_df.xs('p', level='variable')

data_ppc            = data.copy()
data_ppc['p_mean']  = ppc_p.mean(axis=1).values
data_ppc['p_lo']    = ppc_p.quantile(.025, axis=1).values
data_ppc['p_hi']    = ppc_p.quantile(.975, axis=1).values
data_ppc['bin']     = (pd.cut(data_ppc['log(n2/n1)'], 12)
                         .apply(lambda x: x.mid).astype(float))

g_ppc = (data_ppc.groupby(['n1', 'bin'])[['choice', 'p_mean', 'p_lo', 'p_hi']]
                  .mean().reset_index())

import matplotlib.patches as mpatches

def draw_ppc(data, **kwargs):
    ax = plt.gca()
    ax.fill_between(data['bin'], data['p_lo'], data['p_hi'],
                    color='steelblue', alpha=.25, label='95 % CI')
    ax.plot(data['bin'], data['p_mean'], color='steelblue', lw=2, label='Model mean')
    ax.scatter(data['bin'], data['choice'], color='steelblue', s=20, zorder=5, label='Observed')
    ax.axhline(.5, ls='--', c='gray', lw=1)
    ax.axvline(0, ls='--', c='gray', lw=1)
    ax.set_ylim(-.05, 1.05)

g = sns.FacetGrid(g_ppc, col='n1', col_wrap=3, height=3.2, aspect=1.1, sharey=True)
g.map_dataframe(draw_ppc)
g.set_axis_labels('log(n2 / n1)', 'P(chose n2)')
g.set_titles('n₁ = {col_name}')
for ax in g.axes.flat:
    sns.despine(ax=ax)

legend_handles = [
    mpatches.Patch(color='steelblue', alpha=.25, label='95 % CI'),
    plt.Line2D([0], [0], color='steelblue', lw=2, label='Model mean'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue',
               markersize=6, label='Observed'),
]
g.figure.legend(handles=legend_handles, loc='lower center', ncol=3,
                fontsize=9, bbox_to_anchor=(.5, -.04))
g.figure.suptitle('Posterior predictive check — MagnitudeComparisonModel',
                   fontsize=13, y=1.02)
g.figure.tight_layout()